In [10]:
import os
import re
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

con = duckdb.connect()
con.execute("INSTALL postgres; LOAD postgres;")
con.execute(f"""
    ATTACH 'host=localhost port={os.environ['POSTGRES_PORT']} dbname={os.environ['POSTGRES_DB']}
            user={os.environ['POSTGRES_USER']} password={os.environ['POSTGRES_PASSWORD']}'
    AS pg (TYPE postgres, READ_ONLY)
""")

In [11]:
docs = con.sql("""
    SELECT complaint_id,
           ANY_VALUE(make)            AS make,
           ANY_VALUE(model)           AS model,
           STRING_AGG(DISTINCT component, ' | ') AS components,
           ANY_VALUE(complaint_text)  AS text,
           ANY_VALUE(received_date)   AS received_date
    FROM pg.staging.stg_complaints
    WHERE complaint_text IS NOT NULL
    GROUP BY complaint_id
    ORDER BY complaint_id
""").df()

print(f"{len(docs):,} complaints")
docs.head()

429,070 complaints


,complaint_id,make,model,components,text,received_date
0,565713,PLYMOUTH,COLT,SEAT BELTS,"RESPOND TO REQUEST FOR COMPENSATION DUE TO DRIVER RECEIVING INJURIES PRIOR TO RECEIVING NOTICE OF RECALL (95V-103-003, SUMMARY SENT) FOR SEAT BELT BUCKLES IN A 1987 PLYMOUTH COLT AND SENT A VOQ FOR FUTURE USE. CONSUMER SENT ADDITIONAL CORRESPONDENCE. *LD *JS *JS",2020-01-16
1,10816121,PONTIAC,VIBE,AIR BAGS,2007 PONTIAC VIBE. CONSUMER WRITES IN REGARDS TO PASSENGER AIRBAG RECALL. *SMD CONSUMER SENT ADDITIONAL CORRESPONDENCE. *LD THE CONSUMER STATED THE MANUFACTURER EXCEEDED A REASONABLE AMOUNT OF TIME COMPLETING THE RECALL REPAIR. *JS,2020-10-05
2,11289434,HONDA,CR-V,STEERING,2019 HONDA CR-V. CONSUMER WRITES IN REGARDS TO STEERING WHEEL CONTROLS & SRS SUB-WIRE SAFETY RECALL. *LD THE CONSUMER STATED THE MANUFACTURER EXCEEDED A REASONABLE AMOUNT OF TIME COMPLETING THE RECALL REPAIR.*JS,2020-01-02
3,11289436,HYUNDAI,SONATA,VEHICLE SPEED CONTROL,"2018 HYUNDAI SONATA. CONSUMER WRITES IN REGARDS TO UNINTENDED ACCELERATION. *LD THE CONSUMER STATED THE WHILE ATTEMPTING TO MAKE A RIGHT TURN INTO A PARKING SPACE THE VEHICLE ACCELERATED RAPIDLY. THE CONSUMER ATTEMPTED TO STOP THE VEHICLE BY APPLYING THE BRAKES, BUT TO NO AVAIL. THE VEHICLE CRASHED OVER CONCRETE PARKING BARRIERS INTO AN EMBANKMENT WITH SHRUBS AND WAS STOPPED BY A TREE ANGLED UPWARDS. THE DEALER WAS CONTACTED AND DID NOT OFFER ANY ASSISTANCE. *JS",2020-01-02
4,11290552,DODGE,DURANGO,ENGINE,"2011 DODGE DURANGO. CONSUMER WRITES IN REGARDS TO FIRE UNDER THE HOOD OF VEHICLE. *LD THE CONSUMER STATED WHILE DRIVING ALL THE LIGHTS FLASHED ON THE DASHBOARD; THE MOTOR SHUT DOWN, AND SMOKE ERUPTED FROM UNDER THE HOOD. THE CONSUMER WAS ABLE TO COAST TO THE SIDE AT WHICH POINT THE SMOKE BECAME FLAMES, THE VEHICLE WAS TOTALED. *JS*DT*JB",2020-01-02


In [12]:
BOILERPLATE = [
    r"information redacted pursuant to the freedom of information act.{0,40}",
    r"redacted pursuant.{0,60}foia",
    r"\bx{2,}\b",
]
def clean_text(text, make, model):
    t = text.lower()
    t = re.sub(r"tl\*|\*tr", " ", t)
    for word in f"{make} {model}".lower().split():
        t = re.sub(rf"\b{re.escape(word)}\b", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    for pattern in BOILERPLATE:
        t = re.sub(pattern, " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

docs["clean"] = [clean_text(t, mk, md) for t, mk, md in zip(docs["text"], docs["make"], docs["model"])]
docs["n_words"] = docs["clean"].str.split().str.len()
print((docs["n_words"] < 5).sum(), "complaints with fewer than 5 words after cleaning")

4103 complaints with fewer than 5 words after cleaning


In [13]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

custom_stop = {
    "contact", "stated", "owns", "vehicle", "car", "issue", "problem", "time",
    "approximately", "informed", "mph", "miles", "dealer", "dealership",
    "manufacturer", "notified", "repaired", "failure", "number", "vin",
    "independent", "mechanic", "diagnosed", "taken", "referred", "hotline",
    "nhtsa", "mileage", "assistance", "illuminated", "attached", "document",
    "complainant", "complaint", "safety", "driving", "happened",
}

docs_ok = docs[docs["n_words"] >= 5].copy()

vectorizer = TfidfVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | custom_stop),
    min_df=20,
    max_df=0.5,
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    dtype=np.float32,
)
X = vectorizer.fit_transform(docs_ok["clean"])
X.shape

(424967, 50000)

In [14]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import MiniBatchKMeans

lsa = make_pipeline(TruncatedSVD(n_components=150, random_state=42), Normalizer(copy=False))
X_lsa = lsa.fit_transform(X)
print(f"variance kept by 150 components: {lsa[0].explained_variance_ratio_.sum():.1%}")

k = 50
km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=8192, n_init=5, init_size=30_000)
docs_ok["cluster"] = km.fit_predict(X_lsa)

sizes = docs_ok["cluster"].value_counts()
print(f"largest cluster: {sizes.max():,}   smallest: {sizes.min():,}")

variance kept by 150 components: 15.4%
largest cluster: 54,936   smallest: 1,412


In [15]:
terms = vectorizer.get_feature_names_out()
centers_in_word_space = lsa[0].inverse_transform(km.cluster_centers_)
order = centers_in_word_space.argsort()[:, ::-1]

summary = []
for c in range(k):
    members = docs_ok[docs_ok["cluster"] == c]
    top_comp = members["components"].str.split(r" \| ").explode().value_counts().head(2)
    summary.append({
        "cluster": c,
        "size": len(members),
        "top_terms": ", ".join(terms[i] for i in order[c, :10]),
        "top_labels": ", ".join(f"{n} ({v})" for n, v in top_comp.items()),
    })

summary = pd.DataFrame(summary).sort_values("size", ascending=False)
pd.set_option("display.max_colwidth", None)
summary

,cluster,size,top_terms,top_labels
5,5,54936,"stop, lights, times, warning, rear, turn, road, light, drive, highway","ELECTRICAL SYSTEM (13183), ENGINE (10912)"
4,4,27105,"told, said, engine, took, just, recall, drive, new, fix, got","ENGINE (10795), ELECTRICAL SYSTEM (6483)"
38,38,22078,"engine, check engine, check, light, engine light, engine warning, warning, warning light, replaced, power","ENGINE (17307), POWER TRAIN (3671)"
45,45,21590,"recall, remedy, repair, defect, risk, service, vehicles, available, engine, replacement","ENGINE (5697), ELECTRICAL SYSTEM (4982)"
31,31,18125,"brake, brake pedal, brakes, pedal, stop, parking, braking, light, booster, parking brake","SERVICE BRAKES (12174), FORWARD COLLISION AVOIDANCE (2363)"
40,40,17771,"transmission, gear, shifting, shift, gears, speed, issues, hard, truck, drive","POWER TRAIN (15081), ENGINE (2513)"
41,41,14689,"oil, engine, change, consumption, oil consumption, oil change, engine oil, low, burning, excessive","ENGINE (13342), POWER TRAIN (1678)"
44,44,12364,"needed, needed replaced, replaced, aware, warning, determined, engine, warning light, undisclosed, failed","ENGINE (5829), POWER TRAIN (2738)"
6,6,11108,"fuel, pump, fuel pump, tank, fuel tank, recall, engine, gasoline, pressure, pressure fuel","FUEL/PROPULSION SYSTEM (7410), ENGINE (2210)"
22,22,10623,"battery, start, dead, replaced, charge, new, electrical, times, jump, battery replaced","ELECTRICAL SYSTEM (8490), ENGINE (2213)"


In [16]:
summary.sort_values("cluster").to_csv("../docs/nlp_clusters_v1.csv", index=False)
print("saved")

saved
